<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Observability for Flue with Langfuse" sidebarTitle: "Flue" logo: "/images/integrations/flue_icon.svg" description: "Learn how to trace Flue agents and workflows with Langfuse via OpenTelemetry." category: "Integrations" -->

# Trace Flue agents with Langfuse

This notebook shows how to integrate **Langfuse** with **[Flue](https://flueframework.com)** to trace, debug, and evaluate your agents and workflows via OpenTelemetry.

> **What is Flue?**\
> [Flue](https://flueframework.com) is an open-source TypeScript framework for building durable AI agents and workflows. It gives any model a harness-style runtime — sessions, tools, skills, sandboxed filesystem and shell access, and durable execution — so you can write agents once and run them anywhere (Node.js, Cloudflare, CI).

> **What is Langfuse?**\
> [Langfuse](https://langfuse.com) is an open-source LLM engineering platform that helps teams trace, debug, and evaluate their LLM applications.

<!-- STEPS_START -->
## Step 1: Install dependencies

Flue ships a dedicated [`@flue/opentelemetry`](https://github.com/withastro/flue/tree/main/packages/opentelemetry) adapter that turns its public `observe(...)` event stream into OpenTelemetry spans. Pair it with the Langfuse OpenTelemetry SDK ([`@langfuse/otel`](https://langfuse.com/docs/observability/sdk/overview#setup)) to export those spans to Langfuse.

```bash
npm install @flue/runtime @flue/opentelemetry @langfuse/otel @opentelemetry/sdk-node @opentelemetry/api hono valibot
npm install --save-dev @flue/cli
```

> If you don't have a Flue project yet, scaffold one first with `npx flue init --target node` and follow the [Flue quickstart](https://flueframework.com/docs/getting-started/quickstart/).

## Step 2: Configure environment

Get your Langfuse API keys from your project settings in [Langfuse Cloud](https://langfuse.com/cloud) or your [self-hosted](https://langfuse.com/self-hosting) instance, and add them to a `.env` file in your project root together with your model provider key.

```bash
# .env
# Get keys from your project settings: https://langfuse.com/cloud
LANGFUSE_PUBLIC_KEY="pk-lf-..."
LANGFUSE_SECRET_KEY="sk-lf-..."
LANGFUSE_BASE_URL="https://cloud.langfuse.com" # 🇪🇺 EU region
# LANGFUSE_BASE_URL="https://us.cloud.langfuse.com" # 🇺🇸 US region

# Model provider used by your Flue agents
OPENAI_API_KEY="sk-proj-..."
```

## Step 3: Register the Langfuse OpenTelemetry observer

`@flue/opentelemetry` only converts Flue events into spans — your application owns the OpenTelemetry SDK and exporter. Configure the SDK with the Langfuse [`LangfuseSpanProcessor`](https://langfuse.com/docs/observability/sdk/overview#setup) and register the observer in your Flue app's entrypoint (`src/app.ts`).

Flue's model-turn spans follow the [OpenTelemetry GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/), so Langfuse maps them to generations with model, token usage, and cost automatically. The other Flue spans (workflow, operation, tool, task) are emitted under the `@flue/opentelemetry` instrumentation scope, so we extend the default Langfuse span filter to [include that scope](https://langfuse.com/docs/observability/sdk/advanced-features#filtering-by-instrumentation-scope).

```typescript
// src/app.ts
import { NodeSDK } from "@opentelemetry/sdk-node";
import { LangfuseSpanProcessor, isDefaultExportSpan } from "@langfuse/otel";
import { createOpenTelemetryObserver } from "@flue/opentelemetry";
import { observe } from "@flue/runtime";
import { flue } from "@flue/runtime/routing";
import { Hono } from "hono";

const sdk = new NodeSDK({
  spanProcessors: [
    new LangfuseSpanProcessor({
      // Export Langfuse's default LLM spans plus the full Flue span tree
      // (workflow / operation / tool / task) emitted by @flue/opentelemetry.
      shouldExportSpan: ({ otelSpan }) =>
        isDefaultExportSpan(otelSpan) ||
        otelSpan.instrumentationScope.name === "@flue/opentelemetry",
    }),
  ],
});

sdk.start();

// Flush buffered spans on exit: the one-shot `flue run` path (the event
// loop drains) and Ctrl+C on the long-running `flue dev` server (signals).
for (const signal of ["beforeExit", "SIGINT", "SIGTERM"] as const) {
  process.once(signal, async () => {
    await sdk.shutdown();
    if (signal !== "beforeExit") process.exit(0);
  });
}

// Register the observer after sdk.start() so it uses the configured tracer.
observe(createOpenTelemetryObserver());

const app = new Hono();
app.route("/", flue());

export default app;
```

> **Capturing prompts and responses:** by default the adapter exports only metadata (identifiers, durations, model, token usage, and cost) and omits prompts, completions, and tool payloads. To include content, pass an `exportContent` callback — `createOpenTelemetryObserver({ exportContent: (event) => event })` exports everything (use a sanitizing callback in production). Langfuse maps the captured content to the corresponding observation's **Input** and **Output**, surfaces Flue tool calls as dedicated **Tool** observations, and shows delegated tasks as **Agent** observations.

## Step 4: Add an example workflow

Flue discovers workflows from `src/workflows/`. This one runs an agent that uses a tool — each model turn and tool call becomes a span in the trace.

```typescript
// src/workflows/weather.ts
import {
  createAgent,
  defineTool,
  type FlueContext,
  type WorkflowRouteHandler,
} from "@flue/runtime";
import * as v from "valibot";

export const route: WorkflowRouteHandler = async (_c, next) => next();

const agent = createAgent(() => ({ model: "openai/gpt-4o-mini" }));

const lookupWeather = defineTool({
  name: "lookup_weather",
  description: "Look up current weather for a city.",
  parameters: v.object({ city: v.string() }),
  execute: async ({ city }) => `${city}: sunny, 72 F`,
});

export async function run({ init, payload }: FlueContext<{ city?: string }>) {
  const harness = await init(agent);
  const session = await harness.session();
  const city = typeof payload.city === "string" ? payload.city : "San Francisco";
  const response = await session.prompt(
    `Use the weather tool to report current weather in ${city}.`,
    { tools: [lookupWeather] },
  );
  return { message: response.text };
}
```

## Step 5: Run your agent

Start the Flue dev server:

```bash
npx flue dev --target node
```

In another terminal, invoke the workflow and wait for the result:

```bash
curl -X POST "http://localhost:3583/workflows/weather?wait=result" \
  -H "content-type: application/json" \
  -d '{"city":"Berlin"}'
```

You can also run a workflow once without the server with `npx flue run weather --target node --payload '{"city":"Berlin"}'`.

## Step 6: View traces in Langfuse

After running the workflow, open your [Langfuse dashboard](https://langfuse.com/cloud) to see the full trace: the workflow span, each model turn mapped to a generation (with model, token usage, and cost), and tool calls captured as nested **Tool** observations with their inputs and outputs.

[Example trace in Langfuse](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/44bd24918fb224f50b88dc23de15cc01)

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more-js.mdx" -->